# Registry-First Development Workflow with Kiro

Discover agentic capabilities from your organization's AWS Agent Registry, identify gaps, build missing agents, and publish them back — __all from the Kiro IDE__ using MCP-based search and Kiro Powers for publisher workflows.

## What This Tutorial Demonstrates

This tutorial walks through a **registry-first development workflow** powered by Kiro:

1. **Search** — Use the AWS Agent Registry MCP server in Kiro to semantically search for existing agents and tools across your organization, directly from the IDE.

2. **Identify gaps** — Map discovered capabilities against your workflow requirements and pinpoint what's missing.

3. **Build & Publish** — Use Kiro Powers (which define the AWS Agent Registry publisher APIs) to build the missing agents and publish them back to the registry, all via the Kiro chat interface.

4. **Invoke** — Resolve runtime ARNs from registry records and invoke agents via AgentCore Runtime.

The key idea: a developer sitting in Kiro can go from "what agents exist?" to "I built and published the missing ones" without leaving the IDE.

![AWS Agent Registry flow on Kiro](images/KiroDiagram.png)

## What is AWS Agent Registry?

AWS Agent Registry is a centralized, governed catalog for discovering, publishing, and managing AI agents and tools across an organization. It provides semantic search for capability-based discovery, IAM and OAuth access control, rich metadata management (protocol, version, connection info, tool schemas), and a built-in governance workflow (DRAFT → PENDING_APPROVAL → APPROVED) so teams can trust what they find and reuse what others have built. AWS Agent Registry has three pain personas, Publishers - who publish capabilities to the Registry , Admins - who approve the published capabilities and Consumers - who search and access the approved registry records downstream.

![AWS Agent Registry Publisher flow ](images/Publisher-workflow.png)


![AWS Agent Registry Consumer flow](images/2-consumerflow.png) 
## What is Dynamic Client Registration?

Dynamic Client Registration is an OAuth and OpenID Connect protocol that lets client applications automatically register with an authorization server instead of requiring manual pre-registration. The DCR protocol is formally defined in RFC 7591, with optional registration management extensions in RFC 7592. It was designed to work within the Open Authorization (OAuth) and OpenID Connect (OIDC) ecosystems. It enables automatic creation of client IDs, secrets, and metadata (redirect URIs, scopes), often used for automation, AI agents, and dynamic scaling.

### Why DCR Matters for Kiro

For Kiro to search the registry as an MCP tool, the registry needs to be configured with a **CUSTOM_JWT authorizer** backed by an OAuth provider (Auth0 in this example). Kiro's MCP client uses the **authorization_code + PKCE** flow — it dynamically registers itself via Dynamic Client Registration (DCR), opens a browser for login, catches the callback on localhost, and exchanges the code for a token. This means zero manual credential management for the developer.

## How Kiro Powers Fit In this Development Workflow

A **Kiro Power** is a curated, pre-packaged bundle of capabilities designed for the Kiro AI-powered IDE that gives the Kiro agent instant, specialized expertise in a specific technology or workflow.

Each power typically bundles three components :

- **POWER.md** — A steering file that acts as an onboarding manual, telling the agent what MCP tools are available and when to use them
- **MCP server configuration** — The tools and connection details for the Model Context Protocol server
- **Additional steering or hooks** — Extra guidance files or automated validation hooks [ We use this for publisher of records]

The key innovation is **dynamic context loading**: rather than loading every tool upfront (which can overwhelm the agent), powers activate only when relevant. For example, mention "database" and the Neon power loads; switch to deployment and the Netlify power activates while Neon deactivates.

In this sample Kiro Powers package the AWS Agent Registry publisher APIs (`CreateRegistryRecord`, `SubmitRegistryRecordForApproval`, etc.) as steering files that guide Kiro through the publish workflow. When you ask Kiro to "publish this agent to the registry," the Power provides the step-by-step instructions and API calls — so you get a governed publish-and-approve cycle without writing boto3 code yourself.



## Use Case: 

To ground this tutorial in a real world usecase , let imagine a scenrio of AnyCompany financial services.

AnyCompany financial services firm has a multi team structure. Core teams include investment management, wealth advisory, trading operations, and compliance. Over the past year, multiple teams have independently built AI agents, MCP servers, and automation tools — but with no shared catalog, no common standard, and no way for one team to discover what another has already built.

The Wealth Advisory team has been asked to build a Quarterly Intelligence Briefing workflow. 
Here's the requirement: 

> *"When a publicly traded company reports quarterly results, automatically generate a comprehensive client-ready investment brief — including what happened, why it matters, how it affects each client's portfolio, and what (if any) action to consider — all within 30 minutes of the earnings release."*

In order to build this,the key capabilities needed include :
* **First**, gathering data — pull raw earnings data, market context, competitive intel.
* **Second**, analyze and synthesize — run financial analysis compliance tests  and generate an investment thesis based on the data.
* **Third** is generate a perosnlaized investment brief for each client.


The Wealth Advisory team has **zero** existing capabilties in house for earnings data ingestion, financial analysis, or compliance screening. Building from scratch would take **couple of months and $1M+**.

But they don't need to build from scratch. **Other teams already have the pieces.** The Wealth Advisory team just needs a way to find them, verify they're approved for use, and wire them together into a flow.

The result: what would have been a 6-month greenfield project becomes a composition exercise — 7 agents discovered from 5 teams, 2 new agents built to fill gaps, and a single orchestrator tying them all together.




![Wealth Advisory Teams Quarterly Briefing Use Case](images/5-UsecaseviaAWSRegistry.png)



## Tutorial Details

| Information | Details |
|:------------|:--------|
| Tutorial type | Interactive |
| AgentCore components | AWS Agent Registry, AgentCore Runtime, MCP Gateway |
| Record types | MCP, CUSTOM |
| Approval mode | Auto-approval |
| Tutorial components | AWS Agent Registry, AgentCore Runtime, Auth0 (OAuth/DCR), Kiro Powers |
| Tutorial vertical | Financial Services (Wealth Advisory) |
| Example complexity | Advanced |
| SDK used | boto3, strands-agents, mcp |

## Steps Involved: 

Set Up:

- Create an Auth0 DCR-enabled registry (CUSTOM_JWT authorizer) [DCR set up instructions here](https://github.com/awslabs/agentcore-samples/blob/main/01-tutorials/10-Agent-Registry/01-advanced/kiro-registry-dcr-auth0/DCR_registry_search_mcp_in_kiro.ipynb)
- Deploy sample agents to AgentCore Runtime and register them as records

Search

- In Kiro Use the registry MCP server to search for existing agents from chat interface.
- Resolve runtime ARNs from search results and invoke agents from their URIs


Build

- Set up and Use Kiro Powers to create and publish new agents to registry from Kiro Chat interface. [Sample Kiro powers available here ](https://github.com/sanaiqbalw/amazon-bedrock-agentcore-samples/tree/br_dcr-registry_for_kiro-mcp-search/01-tutorials/10-Agent-Registry/01-advanced/kiro/kiro-power-publisher-workflow)

## Prerequisites

- AWS credentials configured with appropriate permissions for AgentCore Registry, AgentCore Runtime, and Cognito
- Python 3.10+ with the following packages:

| Package | Usage |
|:--------|:------|
| `boto3` | AWS SDK for registry and AgentCore runtime interactions |
| `strands-agents` | Agent orchestration framework |
| `mcp` | MCP client for Gateway tool invocation |
| `bedrock-agentcore` | AgentCore Runtime SDK for agent deployment |

- **Kiro IDE** with the AWS Agent Registry MCP server configured and the Agent Registry Publisher Power installed
- **Auth0 tenant** with DCR enabled (for the CUSTOM_JWT authorizer on the registry)

### [Optional — SageMaker AI Only] Create a Machine-to-Machine (M2M) Application

The PKCE flow used by Kiro (and by Step 4 below) requires a localhost HTTP server to catch the OAuth callback. On SageMaker AI, localhost isn't accessible from the browser, so you need to use the **client_credentials** grant instead:

1. In Auth0, navigate to **Applications → Applications**
2. Click **+ Create Application** → select **Machine to Machine Applications**
3. Name it (e.g., `Registry M2M Client`) and authorize it to call the registry API
4. Under **Settings → Credentials**, ensure **Authentication Method** is not `None`
5. Under **Advanced Settings → Grant Types**, ensure **Client Credentials** is enabled


---
# __Step 1:  Registry Setup__

---
## __Step 1.1  Registry Setup__: 
### Create Auth0 DCR Credentials and then create a Registry

Creates registry with CUSTOM_JWT authorizer → polls until READY → adds MCP URL to allowedAudience.


- Follow the steps here to set up Auth0 and create a registry [DCR set up instructions here](https://github.com/awslabs/agentcore-samples/blob/main/01-tutorials/10-Agent-Registry/01-advanced/kiro-registry-dcr-auth0/DCR_registry_search_mcp_in_kiro.ipynb)


- Update .env with domain url and audience 

- Create a registry with auth config using OAUTH DCR created.


In [ ]:
# Create Registry 

from utils import *
from setup import *
result = create_registry_with_auth0(name="AnyCompanyAgentRegistry")
REGISTRY_ID = result["registryId"]

print(f"Registry: {REGISTRY_ID} | Status: {result['status']}")

## __Step 1.2 Registry Setup__ : 
### Add an API on auth0 tenant with identifier as Registry MCP URL


- In the Auth0 dashboard, navigate to **Applications → APIs** and  create an API using the registry's MCP endpoint URL as **Identifier**.Use the registry_id that was created.
mcp url: `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp`

- Note: Kiro sends the MCP server URL as the `audience` in the Auth0 authorization request (Otherwise it wont find the service). In this notebook you do not need to manually update the Auth configurations of the Registry. The `create_registry` helper for this notebook  has been written to automatically add `https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<REGISTRY_ID>/mcp` to the registry's `allowedAudience` after creation. 


---
## __Step 1.3 Registry Setup__:
### Deploy Agents to AgentCore Runtime

In [ ]:
AGENTS_TO_DEPLOY = [
    'data_retrieval_agent',
    'market_research_agent',
    'patent_research_agent',
    'financial_analysis_agent',
    'sentiment_analysis_agent',
    'investment_thesis_generator',
    'approval_workflow_agent',
    'client_communication_agent',
    'reporting_agent',
]

!cd setup && python3 2_deploy_artifacts_on_agentcore_runtime.py {' '.join(AGENTS_TO_DEPLOY)}

---
## __Step 1.4 Registry Setup__:
### Register the deployed Agents as Records

Runtime ARNs are auto-discovered from AgentCore Runtime and embedded in the record descriptors.

In [ ]:

for agent in AGENTS_TO_DEPLOY:
    !cd setup && python3 3_add_records_to_registry.py {agent} --registry-id {REGISTRY_ID}

In [ ]:
# Verify: list all records in the registry
import time
time.sleep(5)
from setup import get_cp_client
cp_client = get_cp_client()

records = cp_client.list_registry_records(registryId=REGISTRY_ID)
print(f"Records ({len(records['registryRecords'])}):\n")
for r in records['registryRecords']:
    print(f" [REGISTRY ID: {REGISTRY_ID}] [{r['status']}] {r['name']} | {r.get('descriptorType', 'N/A')}")


## __Step 2: Search the Registry on Kiro for capabilties__

### This is where your work starts __AS A CONSUMER__ of AWS Agent Registry, when Organization Registry is available

### 2.1 Add AWS Agent Registry mcp to the Kiro mcp.json
Add the Registry mcp url to Kiro mcp.json (.kiro/settings/mcp.json):
{ "mcpServers" : {
"AnyCompanyRegistry": {
      "type": "http",
      "url": "https://bedrock-agentcore.us-west-2.amazonaws.com/registry/<registry_id>/mcp/",
      "disabled": false
    }
}
}


### 2.2 Authenticate

Enable and authenticate the MCP on your Kiro/Claude
When Kiro connects to the MCP server, it will:
- Discover the Auth0 authorization server via the registry's well-known endpoint
- Use DCR to auto-register as an OAuth client (POST /oidc/register)
- Obtain an access token via PKCE authorization code flow
- Use the token to call the registry search MCP


### 2.3 Search from Kiro
Open Kiro chat and ask it to search the registry:

##### prompt 1: 
"Use the AnyCompany Registry to find records relevant for financial data analysis"

##### prompt 2: 
"Use the AnyCompany Registry to find records relevant for complinace workflow tools"

##### prompt 3: 
"Use the AnyCompany Registry to find records relevant for dashboarding and marketing"

##### prompt 4: 
"List all the records that were relevant as a python list "

##### prompt 5: 
"List all the information and inline content of the records above "


#### Then take that python list and invoke the agents to see what they do.

```
queries = [
    "financial data retrieval earnings",
    "market research competitor analysis",
    "patent intellectual property",
    "financial analysis portfolio risk",
    "sentiment analysis news social",
    "approval workflow compliance",
    "reporting dashboard visualization",
    "investment thesis synthesis conviction",
    "client communication personalized brief",
]
```




---
## __Step 3: Resolve Runtime ARNs & Invoke the Agents__

Extract `runtimeArn` from search results metadata in AWS Agent Registry and invoke agents via AgentCore Runtime (IAM auth).

Note: We can do this step in Kiro too, by making sure we have the AgentCore MCP installed.

In [ ]:
from utils import resolve_agents_from_registry, invoke_http_agent

queries=[ "financial data retrieval earnings",
    "market research competitor analysis"] #UPDATE WITH YOUR LIST PLEASE
discovered = resolve_agents_from_registry(queries, registry_id=REGISTRY_ID)

AGENT_ARNS = {n: info['runtimeArn'] for n, info in discovered.items() if info.get('runtimeArn')}

print(f"Resolved {len(AGENT_ARNS)} runtime ARNs:\n")
for name, arn in AGENT_ARNS.items():
    print(f"  {name}: {arn}")



In [ ]:
def invoke_agent(name, prompt):
    """Invoke an HTTP agent by name using its runtime ARN."""
    return invoke_http_agent(AGENT_ARNS[name], prompt)

In [ ]:
# Invoke: Market Research Agent
result = invoke_agent('market_research_agent', 'Analyze the tech sector competitors for NVTK')
print("=== Market Research ===")
print(result['result'][:500])

In [ ]:
# Invoke: Financial Analysis Agent
result = invoke_agent('financial_analysis_agent', 'Analyze NVTK financials')
print("=== Financial Analysis ===")
print(result['result'][:500])

In [ ]:
# Invoke: Sentiment Analysis Agent
result = invoke_agent('sentiment_analysis_agent', 'Analyze sentiment for NVTK')
print("=== Sentiment Analysis ===")
print(result['result'][:500])

In [ ]:
# Invoke: Patent Research Agent
result = invoke_agent('patent_research_agent', 'Analyze NovaTech patent portfolio in AI')
print("=== Patent Research ===")
print(result['result'][:500])

In [ ]:
# Invoke: Investment Thesis Generator
result = invoke_agent('investment_thesis_generator', 'Generate investment thesis for NVTK')
print("=== Investment Thesis ===")
print(result['result'][:500])

In [ ]:
# Invoke: Approval Workflow Agent
result = invoke_agent('approval_workflow_agent', 'Submit investment thesis for compliance review')
print("=== Compliance Approval ===")
print(result['result'][:500])

In [ ]:
# Invoke: Client Communication Agent
result = invoke_agent('data_retrieval_agent', 'Generate growth brief for NVTK thesis')
print("=== Client Brief ===")
print(result['result'][:500])


# __Step 4: Publish New Agents (fill the gaps)__

The Wealth Advisory team builds two new agents:
- `investment_thesis_generator` — synthesizes all analysis into a unified thesis with conviction rating
- `client_communication_agent` — generates personalized briefs per client segment

1. Add Investment Thesis Agent 
2. Add Client Communication Agent


### 4.1 Create Agent Scripts and deploy and register them from Kiro using Kiro powers
Follow the instruction on how to use kiro power to create agents in AgentCore Runtime and publish them.
[Set up kiro powers](https://github.com/sanaiqbalw/amazon-bedrock-agentcore-samples/tree/br_dcr-registry_for_kiro-mcp-search/01-tutorials/10-Agent-Registry/01-advanced/kiro/kiro-power-publisher-workflow)
This will help you in :
1. Deploy the 2 new agents to AgentCore Runtime
2. Deploy the 2 new agents to AgentCore Runtime 
3. Add to Registry, Submit for approval


### 4.2 Go to AWS console in Admin role and approve the agent submission.

______________________





## __Step 5 Verify the published Agents are now discoverable in Registry__

In [ ]:
# Verify they're now discoverable
results = search_registry("investment thesis synthesis conviction", registry_id)
print_results("investment thesis synthesis conviction", results)



**The gaps are filled.** Both agents are now in the registry, discoverable by any team at AnyCompany.

---
## Step 7: Orchestrate — The Full Workflow

Now we compose all 9 agents into the 3-phase Quarterly Intelligence Briefing using Strands Agents.

Each registry agent becomes a `@tool` that the orchestrator can call. The orchestrator is a single Strands `Agent` with a system prompt that describes the 3-phase workflow:

| Phase | Tools | Purpose |
|:------|:------|:--------|
| Phase 1: Gather | `query_earnings`, `market_research`, `patent_research` | Collect raw data (parallel) |
| Phase 2: Analyze & Synthesize | `financial_analysis`, `sentiment_analysis`, `generate_thesis`, `compliance_review` | Deep analysis and thesis generation (sequential) |
| Phase 3: Deliver | `generate_client_brief`, `generate_report` | Personalized briefs and formatted reports |


In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel

# ── PHASE 1 tools: GATHER ──

@tool
def query_earnings(ticker: str, quarter: str) -> str:
    """Retrieve quarterly earnings data."""
    return invoke_http_agent(AGENTS['data_retrieval_agent'], f'Get earnings data for {ticker} {quarter}')['result']

@tool
def market_research(ticker: str) -> str:
    """Analyze market context and competitors for a ticker."""
    return invoke_http_agent(AGENTS['market_research_agent'], f'Analyze the tech sector competitors for {ticker}')['result']

@tool
def patent_research(company: str) -> str:
    """Analyze patent portfolio and IP moats."""
    return invoke_http_agent(AGENTS['patent_research_agent'], f'Analyze {company} patent portfolio in AI')['result']

# ── PHASE 2 tools: ANALYZE & SYNTHESIZE ──

@tool
def financial_analysis(ticker: str) -> str:
    """Perform financial analysis including valuation and risk metrics."""
    return invoke_http_agent(AGENTS['financial_analysis_agent'], f'Analyze {ticker} financials')['result']

@tool
def sentiment_analysis(ticker: str) -> str:
    """Analyze sentiment from news and social media."""
    return invoke_http_agent(AGENTS['sentiment_analysis_agent'], f'Analyze sentiment for {ticker}')['result']

@tool
def generate_thesis(ticker: str, financial_summary: str, sentiment_summary: str, market_context: str) -> str:
    """Synthesize all analysis into a unified investment thesis with conviction rating."""
    prompt = f'Generate investment thesis for {ticker}. Financials: {financial_summary[:500]}. Sentiment: {sentiment_summary[:500]}. Market: {market_context[:500]}'
    return invoke_http_agent(AGENTS['investment_thesis_generator'], prompt)['result']

@tool
def compliance_review(thesis: str) -> str:
    """Submit investment thesis for compliance approval."""
    return invoke_http_agent(AGENTS['approval_workflow_agent'], f'Submit investment thesis for compliance review: {thesis[:500]}')['result']

# ── PHASE 3 tools: DELIVER ──

@tool
def generate_client_brief(thesis: str, client_segment: str) -> str:
    """Generate personalized client brief for a segment (conservative/growth/institutional)."""
    return invoke_http_agent(AGENTS['client_communication_agent'], f'Generate {client_segment} brief for thesis: {thesis[:500]}')['result']

@tool
def generate_report(title: str, content: str) -> str:
    """Format and generate a report."""
    return invoke_http_agent(AGENTS['reporting_agent'], f'Generate report titled {title}: {content[:500]}')['result']

print('All 9 workflow tools defined.')

In [ ]:
# Create the orchestrator agent
orchestrator = Agent(
    model=BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0'),
    tools=[
        query_earnings, market_research, patent_research,       # Phase 1: Gather
        financial_analysis, sentiment_analysis,                  # Phase 2a: Analyze
        generate_thesis, compliance_review,                      # Phase 2b: Synthesize
        generate_client_brief, generate_report,                  # Phase 3: Deliver
    ],
    system_prompt="""You are the Quarterly Intelligence Briefing orchestrator for AnyCompany Financial Services.

When given a ticker and quarter, execute this 3-phase workflow:

PHASE 1 — GATHER (call these in parallel):
  1. query_earnings — get raw earnings data
  2. market_research — get market context and competitor analysis
  3. patent_research — get IP and competitive moat analysis

PHASE 2 — ANALYZE & SYNTHESIZE (sequential):
  4. financial_analysis — deep financial analysis
  5. sentiment_analysis — news and social sentiment
  6. generate_thesis — synthesize everything into a unified investment thesis
  7. compliance_review — submit thesis for compliance approval

PHASE 3 — DELIVER:
  8. generate_client_brief — create personalized briefs for 'growth' segment
  9. generate_report — format the final report

After all phases, provide a summary of the complete briefing."""
)

print('Orchestrator ready.')

In [ ]:
# Run the full Quarterly Intelligence Briefing
result = orchestrator('Generate a Quarterly Intelligence Briefing for NVTK Q1-2025')
display(Markdown(f'## Quarterly Intelligence Briefing — NVTK Q1-2025\n\n{str(result)}'))

---
## Step 8:  to Cleanup

Deletes all records, the registry, discovered runtimes, gateways, OAuth providers, and Cognito pools.

In [ ]:
cleanup(REGISTRY_ID)